<a href="https://colab.research.google.com/github/100522401/P1_AA_100522396_100522401/blob/main/notebooks/Cuaderno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica 1 NIA 100522396 NIA 100522401


# Introducción



El objetivo de la práctica es **construir un modelo** de aprendizaje automático capaz de predecir si un cliente bancario suscribirá un depósito a plazo. Para construir ese modelo nos basaremos en la información demográfica, financiera e interacciones de los clientes.

Como variable objetivo del problema tenemos 'deposit'. Esta variable indicará si el cliente contrató, o no, el producto ofrecido. Por tanto, se trata de un problema de **clasificación binaria**.
<br><br>
En este cuaderno se incluye:

  - Exploratory Data Analysis (EDA).
  - Definir una estrategia de evaluación.
  - Optimización de hiperparámetros (HPO).
  - Selección del mejor modelo y estudio de su rendimiento.
  - Entrenamiento del modelo final.

## Configuración inicial y carga de datos

En esta sección se importan las librerías básicas usadas, se fija la semilla con el nía 1005224**01** y se cargan los datos desde Google Drive

En primer lugar, se importan las librerías básicas y fijamos la semilla aleatoria.

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 1    #100522401
np.random.seed(SEED)

Cargamos el dataset de entrenamiento desde Google Drive

In [23]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/P1_AA_datasets"
TRAIN_PATH = f"{BASE_DIR}/bank_01.pkl"
# COMP_PATH  = f"{BASE_DIR}/bank_competition.pkl"

print("TRAIN_PATH:", TRAIN_PATH)
# print("COMP_PATH:", COMP_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TRAIN_PATH: /content/drive/MyDrive/P1_AA_datasets/bank_01.pkl


Se comprueba que el dataset se ha cargado correctamente

In [24]:
#TODO: Se podrían añadir try/excepts al read...

data = pd.read_pickle(TRAIN_PATH)
display(data.head())
print("\nDatos cargados correctamente.")

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes



Datos cargados correctamente.


# EDA

En esta sección se realiza un análisis exploratorio de los datos (EDA) básico.

En este estudio determinaremos:

- Número de instancias.
- Número de variables ( incluyendo la variable objetivo).
- Identificación de variables numéricas, categóricas u ordinales.
- Estudio de cardinalidad.
- Análisis de la variable 'pdays'.
- Estudio de los valores faltantes, constantes o identificadores.
- Estudio del problema y análisis de desbalanceos de la variable objetivo.


En este bloque se calcula el tamaño del dataset (número de instancias y variables). También se verfiica que la variable objetivo existe. Se comprueba cuántos valores únicos tienen deposit para confirmar el tipo de problema.

In [25]:
#Tamaño del dataset y tipo de problema
n_rows, n_cols = data.shape
print(f"Instancias: {n_rows}")
print(f"Variables totales (incluyendo target): {n_cols}")

target = "deposit"
if target not in data.columns:
    raise ValueError(f"No existe la variable objetivo '{target}' en el dataset.")

n_unique_target = data[target].nunique(dropna=False)
print(f"Variable objetivo: {target}")
print(f"Valores únicos en target: {n_unique_target}")

# La variable 'deposit' indica si suscribe o no: problema de clasificación binaria.
if n_unique_target == 2:
    problem_type = "clasificación binaria"
else:
    problem_type = "clasificación (revisar codificación de clases)"

print(f"Tipo de problema: {problem_type}")

Instancias: 11000
Variables totales (incluyendo target): 17
Variable objetivo: deposit
Valores únicos en target: 2
Tipo de problema: clasificación binaria


En este caso tenemos dos clases, y por lo tanto se concluye que es un problema de **clasificación binaria** (se suscribe o no se suscribe). Es importante tener esto en cuenta ya que condiciona la elección de modelos y métricas en el resto de la práctica.

En este bloque se identifican los tipos de variables, separandolas en numéricas (donde se incluyen als booleanas) y categóricas. Esta distinción nos ayudará a definir el preprocesado, ya que las variables numéricas y categóricas requieren transformaciones distintas.

In [26]:
# 2) Tipos de variables
X = data.drop(columns=[target])

num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numéricas: {len(num_cols)}")
print(num_cols)
print("\nCategóricas: {0}".format(len(cat_cols)))
print(cat_cols)

Numéricas: 7
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Categóricas: 9
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


Se han identificado 7 variables numéricas y 9 varaibels catagóricas. Esta separación nos sirve para contruir las pipelines de preprocesado, aplicando transformaciones especificas a cada tipo de variable.

Ahora calcularemos la cardinalidad (número de valores únicos) de las variables categóricas para detectar posibles casos de alta cardinalidad. Es importante tenerlo en cuenta porque al aumentar la dimensionalidad del conjunto de datos, aumenta el coste computacional y el rendimiento del modelo.

In [27]:
# 3) Cardinalidad en variables categóricas
if len(cat_cols) == 0:
    print("No hay variables categóricas.")
else:
    cardinality = X[cat_cols].nunique(dropna=False).sort_values(ascending=False)
    display(cardinality.to_frame("n_unique"))

    high_card_cols = cardinality[cardinality > 10].index.tolist()
    if high_card_cols:
      print("\nCategóricas con alta cardinalidad (>10):", high_card_cols)
    else:
      print("\nCategóricas con alta cardinalidad (>10): ninguna")

,n_unique
job,13
month,12
education,5
poutcome,4
marital,3
contact,3
default,2
housing,2
loan,2



Categóricas con alta cardinalidad (>10): ['job', 'month']


En este caso, se observa que hay dos columnas que superan el umbral de alta caridnalidad (> 10 valores únicos).

A continuación analizaremos los valores faltantes del dataset, calculando para cada variable cuántos valores nulos aparecen y qué porcentaje representan. Esta información es importante para decidir quÉ estrategia se utilizará en el preprocesado (imputación)

In [28]:
# 4) Valores faltantes
missing_count = data.isna().sum().sort_values(ascending=False)
missing_ratio = (missing_count / len(data) * 100).round(2)
missing_df = pd.DataFrame({"n_missing": missing_count, "pct_missing": missing_ratio})
missing_df = missing_df[missing_df["n_missing"] > 0]

if missing_df.empty:
    print("No hay valores faltantes.")
else:
    display(missing_df)


,n_missing,pct_missing
education,316,2.87
job,169,1.54


Se observa que hay dos varibales con valores faltantes. Estas variables requerirán una estrategia de imputación en el preprocesado, que constará en el pipeline. Esto evitará problemas durante el entrenamiento de los modelos.

Se buscan posibles comunas problemáticas para el modelado: columnas constantes y columnas tipo ID (con valores casi únicos o identificables). ESte tipo de variables aporta poca capacidad predicitiva y puede introducir ruido o sobreajusnte, por lo que es mejor detectarlas antes del preprocesado.

In [29]:
# 5) Columnas constantes y posibles ID
nunique = X.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()

id_like_by_name = [c for c in X.columns if "id" in c.lower()]
id_like_by_uniqueness = [c for c in X.columns if X[c].nunique(dropna=False) >= 0.98 * len(X)]
id_like_cols = sorted(set(id_like_by_name + id_like_by_uniqueness))

print("Columnas constantes:", constant_cols if constant_cols else "ninguna")
print("Posibles columnas ID:", id_like_cols if id_like_cols else "ninguna")

Columnas constantes: ninguna
Posibles columnas ID: ninguna


No se han detectado **columnos constantes** ni **posibles columnas tipo ID** en el dataset. Por tanto, no es necesario eliminar variables por estos motivos en el preprocesao y modelado.

Ahora analizaremos la distribución de la variable objetivo **deposit**, ya que en problemas de clasificación el desbalanceo relevante relamente es el de las clases que se buscan predecir. ESta información nos será útil para justificar la elección de la métrica y la comparación entre modelos.

En este bloque se analiza la variable **pdays**, la cual representa el númerod e dúas desde el último contacto en la campaña anterior y tomará el valor **-1** cuando el cliente no ha sido contactado previamente. ENtonces, calcularemos la proporcion de valores -1 y se describen los valores válidos (pdays!=-1) para decidir como tratar la variable en el preprocesado.

In [30]:
"""
# Desbalanceo del target

# Conteo y porcentaje por clase
print("Conteo por clase:")
print(counts)

print("\nPorcentajes (%):")
print(perc.round(2))

# Ratio de desbalanceo (mayor/minor)
imbalance_ratio = counts.max() / counts.min()
print("\nImbalance ratio (mayor/minor):", round(imbalance_ratio, 2))

# Gráfico
ax = counts.plot(kind="bar")
plt.title("Distribución del target (deposit)")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.xticks(rotation=0)
plt.show()
"""

'\n# Desbalanceo del target\n\n# Conteo y porcentaje por clase\nprint("Conteo por clase:")\nprint(counts)\n\nprint("\nPorcentajes (%):")\nprint(perc.round(2))\n\n# Ratio de desbalanceo (mayor/minor)\nimbalance_ratio = counts.max() / counts.min()\nprint("\nImbalance ratio (mayor/minor):", round(imbalance_ratio, 2))\n\n# Gráfico\nax = counts.plot(kind="bar")\nplt.title("Distribución del target (deposit)")\nplt.xlabel("Clase")\nplt.ylabel("Frecuencia")\nplt.xticks(rotation=0)\nplt.show()\n'

La variabel obketivo presneta una distribución de **52,55%** para la clase **no** y **47,45%** para la clase **yes**. Por tanto, el problema puedoe considerarse prácticamente equilibrado, con un desbalanceo poco notorio.

In [31]:
# Analisis de pdays
p = data["pdays"]

n_total = len(p)
n_minus1 = (p == -1).sum()
pct_minus1 = 100 * n_minus1 / n_total

print(f"Total: {n_total}")
print(f"pdays = -1 (sin contacto previo): {n_minus1} ({pct_minus1:.2f}%)")

p_valid = p[p != -1]
print("\nResumen pdays (solo contactos previos reales):")
n_notminus1 = (p != -1).sum()
pct_notminus1 = 100 * n_notminus1 / n_total
print(f"pdays != -1 (solo contactos previos reales): {n_notminus1} ({pct_notminus1:.2f}%)")

display(p_valid.describe())



Total: 11000
pdays = -1 (sin contacto previo): 8203 (74.57%)

Resumen pdays (solo contactos previos reales):
pdays != -1 (solo contactos previos reales): 2797 (25.43%)


,pdays
count,2797.000000
mean,204.718627
std,122.397953
min,1.000000
25%,98.000000
50%,182.000000
75%,286.000000
max,854.000000


Se observa que **pdays == -1** aparece en **8203 de 11000 instancias (74.57%)**, lo que indica que en al mayoría de los casos no existe un contacto previo. Por tanto, el valor -1 no debe interpretarse como un valor numérico, ya que no aporta informaciónr relevante al modelado. Para los casos de **pdays != -1**, el número de días tiene una distribución amplia ( mediana **182**, rango **1 - 854**), lo que aporta información que puede ser útil para el modelado.

En este bloque se dine el preprocesado de **pdays**. DAdo que **pdays == -1** no represnta un número de días real, sino "sin contacto previo", separemos esta información en una **variable binaria (contacted_before)** y se trasnforma pdays para que contenga únicamente valores numéricos válidos, reemplazando -1 por NaN. De este modo se evita que -1 distorsione el escalado de la variable.

In [32]:
# Preprocesado pdays


# Creo nueva columna 0: no contactado previamente 1: contactado previamente
X['contacted_before'] = (X['pdays'] != -1).astype(int)

# Remplazo -1 de pdays por Nan
X["pdays"] = X["pdays"].replace(-1, np.nan)

X.head(300)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,contacted_before
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,NaN,0,unknown,0
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,NaN,0,unknown,0
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,NaN,0,unknown,0
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,NaN,0,unknown,0
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,NaN,0,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,36,blue-collar,divorced,secondary,no,638,yes,no,unknown,16,jun,1395,2,NaN,0,unknown,0
301,48,unemployed,single,tertiary,no,3229,no,no,unknown,16,jun,1089,1,NaN,0,unknown,0
302,51,management,single,secondary,no,1610,yes,no,unknown,16,jun,1248,3,NaN,0,unknown,0
303,51,blue-collar,married,secondary,no,33,no,yes,unknown,16,jun,721,3,NaN,0,unknown,0


## 3. Esquema de evaluación

In [40]:
from sklearn.model_selection import train_test_split, StratifiedKFold
#Todas las columnas menos la target
X = data.drop(columns='deposit')
#Solo columna target
y = data['deposit']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=1/3,
    stratify=y,
    random_state=SEED
)

print("Tamaño total de X:", X.shape)
print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)

print("\nDistribución de clases en y:")
print(y.value_counts(normalize=True))

print("\nDistribución de clases en y_train:")
print(y_train.value_counts(normalize=True))

print("\nDistribución de clases en y_test:")
print(y_test.value_counts(normalize=True))

cv_inner = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

print("\n",cv_inner)

Tamaño total de X: (11000, 16)
Tamaño de X_train: (7333, 16)
Tamaño de X_test: (3667, 16)

Distribución de clases en y:
deposit
no     0.525455
yes    0.474545
Name: proportion, dtype: float64

Distribución de clases en y_train:
deposit
no     0.525433
yes    0.474567
Name: proportion, dtype: float64

Distribución de clases en y_test:
deposit
no     0.525498
yes    0.474502
Name: proportion, dtype: float64

 StratifiedKFold(n_splits=5, random_state=1, shuffle=True)


Se utilizará el **F1-score** como métrica de evaluación, ya que el problema es de clasificación binaria y esta métrica es más adecuada que la accuracy cuando existe desbalanceo entre clases. El F1-score combina precisión y recall, permitiendo valorar mejor el comportamiento del modelo.